# NB_03 — Process-to-Response Validation Plan v1.0

**Engineering question**

> Which replicated electroplating measurements are required to convert the current `measurement_plan_ready` state into evidence-supported numerical process specifications?

This notebook does **not** invent manufacturing tolerances or synthetic measurements. It converts the unresolved NB_02 process dimensions into a reproducible validation design and a schema for future process-to-response evidence.

The current evidence state is:

- three source-supported electroplating operating points;
- five unresolved process dimensions;
- no validated numerical process window;
- no candidate numerical specifications yet.

The notebook therefore produces a **measurement contract**, not a numerical answer.


## Workflow

```text
NB_02 process-window status
        +
Engineering Objects
        +
SOURCE_01–SOURCE_04
        ↓
unresolved dimensions
        ↓
validation-factor matrix
        ↓
response-measurement matrix
        ↓
replication / batch plan
        ↓
future evidence schema
        ↓
readiness gates
        ↓
NB_03 export
```


## 1. Locate repository and load current engineering state


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None

NOTEBOOK_ID = "NB_03_PROCESS_TO_RESPONSE_VALIDATION_PLAN"
VALIDATION_ID = "VALIDATION_PLAN_01"
PROCESS_WINDOW_ID = "PROCESS_WINDOW_01"

SOURCE_FILES = [
    "SOURCE_01_bismuth_microstructure.yaml",
    "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
    "SOURCE_03_electroplating_process.yaml",
    "SOURCE_04_thermal_conductivity.yaml",
]


def find_repo_root() -> Path:
    candidates = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])
    candidates.extend([
        Path("/content/sensors-becker"),
        Path("/home/dan/sensors-becker"),
        Path.home() / "sensors-becker",
    ])

    for candidate in candidates:
        if candidate.is_dir() and (candidate / "engineering_navigator").is_dir():
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if not target.exists():
            subprocess.run(["git", "clone", REPOSITORY_URL, str(target)], check=True)
        if (target / "engineering_navigator").is_dir():
            return target

    raise FileNotFoundError(
        "Could not locate sensors-becker. Set REPO_ROOT_OVERRIDE explicitly."
    )


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OBJ_DIR = ROOT / "engineering_navigator" / "engineering_objects"
SOURCE_DIR = (
    ROOT
    / "engineering_navigator"
    / "absorber_manufacturing"
    / "source_records"
)
PROCESS_OUTPUT_DIR = (
    ROOT
    / "outputs"
    / "engineering_questions"
    / "absorber_manufacturing"
    / PROCESS_WINDOW_ID
)
OUTPUT_DIR = (
    ROOT
    / "outputs"
    / "engineering_questions"
    / "absorber_manufacturing"
    / VALIDATION_ID
)
EXPORT_DIR = ROOT / "exports" / VALIDATION_ID
EXPORT_ZIP = ROOT / "exports" / f"{VALIDATION_ID}_export.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

print(f"Repository : {ROOT}")


## 2. Load Engineering Objects and completed source records


In [ ]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)

    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise TypeError(f"{path}: expected one top-level YAML mapping")

    return data


engineering_objects = {
    object_id: load_yaml(OBJ_DIR / f"{object_id}.yaml")
    for object_id in ("absorber", "electroplating", "tes")
}

records = {}

for filename in SOURCE_FILES:
    record = load_yaml(SOURCE_DIR / filename)
    source_id = record.get("source_id")

    if not source_id:
        raise KeyError(f"{filename}: missing source_id")

    if source_id in records:
        raise ValueError(f"Duplicate source_id: {source_id}")

    records[source_id] = record


incomplete = {
    source_id: record.get("extraction_status")
    for source_id, record in records.items()
    if not str(record.get("extraction_status", "")).startswith("complete")
}

if incomplete:
    raise ValueError(
        "Incomplete source records: " + json.dumps(incomplete, indent=2)
    )

print("Engineering Objects:", ", ".join(engineering_objects))
print("Source-record validation: PASS")
print("Sources:", ", ".join(sorted(records)))


## 3. Resolve the current NB_02 state

If NB_02 output files are present, this notebook reads them directly. In a fresh clone, it falls back to the known unresolved process dimensions encoded by the current Engineering Object / process-window architecture.

This fallback does **not** supply numerical tolerances; it only preserves the validation contract.


In [ ]:
status_path = PROCESS_OUTPUT_DIR / "process_window_status.json"
window_path = PROCESS_OUTPUT_DIR / "candidate_process_window.csv"
validation_path = PROCESS_OUTPUT_DIR / "validation_matrix.csv"

if status_path.exists():
    process_window_status = json.loads(status_path.read_text(encoding="utf-8"))
else:
    process_window_status = {
        "status": "measurement_plan_ready",
        "numeric_process_window_established": False,
        "source_supported_operating_points": len(
            engineering_objects["electroplating"].get("reported_process_points", [])
        ),
        "leading_process_variables": [
            "Bi_thickness",
            "grain_size",
            "current_density",
            "bias_voltage",
            "plating_rate",
        ],
        "leading_detector_responses": [
            "low_energy_tail_fraction",
            "quantum_efficiency",
            "energy_resolution",
            "C",
            "G",
        ],
        "unresolved_process_dimensions": 5,
        "next_engineering_step": (
            "Validate replicated process-to-response measurements "
            "and populate numerical candidate ranges."
        ),
    }

if process_window_status.get("numeric_process_window_established"):
    raise ValueError(
        "NB_03 v1.0 is designed for the unresolved measurement-plan state."
    )

print(json.dumps(process_window_status, indent=2))


## 4. Validation factors


In [ ]:
factor_rows = [
    {
        "factor": "Bi_thickness",
        "engineering_role": "stopping power + thermalization",
        "study_type": "controlled sweep",
        "required_replication": "multiple devices per thickness, repeated batches",
        "held_constant": "electroplating chemistry + TES + membrane geometry",
    },
    {
        "factor": "current_density",
        "engineering_role": "electroplating process control",
        "study_type": "controlled sweep around source-supported conditions",
        "required_replication": "multiple films/devices per setting, repeated batches",
        "held_constant": "target thickness + bath chemistry + geometry",
    },
    {
        "factor": "bias_voltage",
        "engineering_role": "electroplating process control",
        "study_type": "controlled sweep where independently controllable",
        "required_replication": "multiple films/devices per setting",
        "held_constant": "target thickness + bath chemistry + geometry",
    },
    {
        "factor": "plating_rate",
        "engineering_role": "deposition kinetics + microstructure",
        "study_type": "controlled sweep / measured-response study",
        "required_replication": "multiple films/devices per rate condition",
        "held_constant": "target thickness + bath chemistry + geometry",
    },
    {
        "factor": "grain_size",
        "engineering_role": "carrier thermalization + spectral tail",
        "study_type": "acceptance-threshold study",
        "required_replication": "distribution across replicated absorbers",
        "held_constant": "nominal absorber thickness + detector design",
    },
]

validation_factors = pd.DataFrame(factor_rows)
validation_factors


## 5. Response measurements


In [ ]:
response_rows = [
    {
        "response": "grain_size_distribution",
        "category": "absorber_state",
        "measurement": "SEM + diffraction",
        "purpose": "connect process settings to microstructure",
    },
    {
        "response": "low_energy_tail_fraction",
        "category": "detector_response",
        "measurement": "TES x-ray spectrum",
        "purpose": "test incomplete thermalization / trapping",
    },
    {
        "response": "quantum_efficiency",
        "category": "detector_response",
        "measurement": "x-ray absorption / count response",
        "purpose": "measure stopping-power benefit of thickness",
    },
    {
        "response": "energy_resolution",
        "category": "detector_response",
        "measurement": "TES x-ray spectrum",
        "purpose": "verify detector performance under process changes",
    },
    {
        "response": "thermal_conductivity_or_thermalization",
        "category": "detector_thermal_design",
        "measurement": "dedicated thermal structure and/or detector timing",
        "purpose": "verify absorber heat reaches the TES before membrane escape",
    },
    {
        "response": "C",
        "category": "detector_thermal_design",
        "measurement": "TES thermal characterization",
        "purpose": "track absorber/TES heat-capacity coupling",
    },
    {
        "response": "G",
        "category": "detector_thermal_design",
        "measurement": "TES thermal characterization",
        "purpose": "track thermal-conductance coupling",
    },
    {
        "response": "yield",
        "category": "manufacturing",
        "measurement": "pass/fail count by wafer/batch",
        "purpose": "convert a feasible recipe into a scalable process specification",
    },
]

response_measurements = pd.DataFrame(response_rows)
response_measurements


## 6. Process-to-response measurement matrix


In [ ]:
matrix_rows = [
    {
        "factor": "Bi_thickness",
        "primary_responses": "low_energy_tail_fraction; quantum_efficiency",
        "secondary_responses": "energy_resolution; C; thermalization",
        "minimum_evidence_needed": "controlled thickness series across replicated devices",
    },
    {
        "factor": "current_density",
        "primary_responses": "grain_size_distribution",
        "secondary_responses": "low_energy_tail_fraction; energy_resolution; yield",
        "minimum_evidence_needed": "controlled current-density series with replicated films/devices",
    },
    {
        "factor": "bias_voltage",
        "primary_responses": "grain_size_distribution",
        "secondary_responses": "morphology; low_energy_tail_fraction; yield",
        "minimum_evidence_needed": "independently measured voltage/process-response series",
    },
    {
        "factor": "plating_rate",
        "primary_responses": "grain_size_distribution",
        "secondary_responses": "roughness; low_energy_tail_fraction; yield",
        "minimum_evidence_needed": "replicated rate/process-response measurements",
    },
    {
        "factor": "grain_size",
        "primary_responses": "low_energy_tail_fraction",
        "secondary_responses": "energy_resolution; repeatability",
        "minimum_evidence_needed": "grain-size distributions linked to detector spectra",
    },
    {
        "factor": "batch",
        "primary_responses": "repeatability; yield",
        "secondary_responses": "grain_size_distribution; low_energy_tail_fraction; thermalization",
        "minimum_evidence_needed": "multiple wafers/batches under nominal recipe",
    },
]

process_response_matrix = pd.DataFrame(matrix_rows)
process_response_matrix


## 7. Future evidence-record schema

Each new replicated measurement set should be ingestible as a source/evidence record rather than pasted directly into the process-window notebook.

The table below defines the minimum fields NB_03 expects for a process-to-response dataset.


In [ ]:
future_evidence_schema = pd.DataFrame([
    {"field": "measurement_id", "required": True, "description": "unique measurement/run identifier"},
    {"field": "batch_id", "required": True, "description": "wafer or fabrication batch identifier"},
    {"field": "device_id", "required": True, "description": "device/sample identifier"},
    {"field": "process_variable", "required": True, "description": "controlled or observed process input"},
    {"field": "process_value", "required": True, "description": "numeric or categorical process value"},
    {"field": "process_unit", "required": True, "description": "unit for process value"},
    {"field": "held_constant", "required": True, "description": "controlled conditions"},
    {"field": "response_variable", "required": True, "description": "measured absorber/detector response"},
    {"field": "response_value", "required": True, "description": "measured response"},
    {"field": "response_unit", "required": True, "description": "unit for response"},
    {"field": "measurement_method", "required": True, "description": "SEM, diffraction, TES spectrum, thermal test, etc."},
    {"field": "source_or_dataset", "required": True, "description": "provenance for literature or new experiment"},
])

future_evidence_schema


## 8. Readiness gates for numerical specifications


In [ ]:
readiness_gates = pd.DataFrame([
    {
        "gate": "replication",
        "criterion": "multiple devices and repeated batches exist for each proposed bounded factor",
        "current_state": "open",
    },
    {
        "gate": "process_to_microstructure",
        "criterion": "process-variable changes map to grain-size/morphology distributions",
        "current_state": "open",
    },
    {
        "gate": "microstructure_to_spectral_response",
        "criterion": "grain-size/morphology distributions map to tail fraction and resolution",
        "current_state": "open",
    },
    {
        "gate": "thickness_tradeoff",
        "criterion": "thickness series jointly measures stopping power and spectral/thermal response",
        "current_state": "open",
    },
    {
        "gate": "thermalization",
        "criterion": "absorber thermalization remains adequate across candidate process conditions",
        "current_state": "open",
    },
    {
        "gate": "yield_repeatability",
        "criterion": "candidate bounds reproduce across wafers/batches with measured yield",
        "current_state": "open",
    },
])

readiness_gates


## 9. Validation-plan status


In [ ]:
validation_status = {
    "notebook_id": NOTEBOOK_ID,
    "status": "validation_contract_ready",
    "upstream_process_window_status": process_window_status.get("status"),
    "numeric_process_window_established": False,
    "source_supported_operating_points": int(
        process_window_status.get("source_supported_operating_points", 0)
    ),
    "validation_factor_count": int(len(validation_factors)),
    "response_measurement_count": int(len(response_measurements)),
    "readiness_gate_count": int(len(readiness_gates)),
    "open_readiness_gate_count": int(
        (readiness_gates["current_state"] == "open").sum()
    ),
    "next_engineering_step": (
        "Add replicated process-to-response evidence records, then rerun NB_02 "
        "and NB_03 to evaluate whether numerical candidate ranges are supported."
    ),
}

validation_status


## 10. Write outputs


In [ ]:
factors_csv = OUTPUT_DIR / "validation_factors.csv"
responses_csv = OUTPUT_DIR / "response_measurements.csv"
matrix_csv = OUTPUT_DIR / "process_response_matrix.csv"
schema_csv = OUTPUT_DIR / "future_evidence_schema.csv"
gates_csv = OUTPUT_DIR / "readiness_gates.csv"
status_json = OUTPUT_DIR / "validation_plan_status.json"

validation_factors.to_csv(factors_csv, index=False)
response_measurements.to_csv(responses_csv, index=False)
process_response_matrix.to_csv(matrix_csv, index=False)
future_evidence_schema.to_csv(schema_csv, index=False)
readiness_gates.to_csv(gates_csv, index=False)

status_json.write_text(
    json.dumps(
        validation_status,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

written_files = {
    "validation_factors": factors_csv,
    "response_measurements": responses_csv,
    "process_response_matrix": matrix_csv,
    "future_evidence_schema": schema_csv,
    "readiness_gates": gates_csv,
    "validation_plan_status": status_json,
}

for name, path in written_files.items():
    print(f"{name:26} {path.relative_to(ROOT)}")


## 11. Build and download export ZIP


In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in written_files.values():
    shutil.copy2(path, EXPORT_DIR / path.name)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## 12. Handoff

NB_03 v1.0 turns the current unresolved process window into a reproducible **validation contract**.

It intentionally stops before numerical optimization. The next source-level advance should be one of:

1. an additional published source containing replicated electroplating process-to-response measurements; or
2. a repository-native experimental dataset recorded using `future_evidence_schema.csv`.

Once such evidence exists:

```text
new evidence record(s)
        ↓
Engineering Object refresh
        ↓
NB_01 synthesis
        ↓
NB_02 process-window update
        ↓
NB_03 readiness-gate evaluation
        ↓
candidate numeric specification
```

*Admissible generalizations trail leading specifications.*
